# 🤖 VisionTracker Remote Server — Kaggle Setup

This notebook sets up a self-hosted **Gemma 3 4B** vision-language model as a FastAPI server for VisionTracker object identification, running on Kaggle's free GPU.

## Why Kaggle?
- 🚀 **More GPU RAM** — 30GB vs Colab's 16GB
- ⏱️ **Longer sessions** — up to 12 hours per week
- 🆓 **Free tier** — no credit card required

## Prerequisites
1. Create a free Kaggle account at https://www.kaggle.com
2. Create a free ngrok account at https://dashboard.ngrok.com/signup
3. Copy your ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
4. Add it as a Kaggle secret: Right sidebar → Add-ons → Secrets → Add secret: `NGROK_AUTHTOKEN`

## Cell 1: Verify GPU & Install Dependencies

Check GPU availability and install required packages.

In [ ]:
# Check GPU
!nvidia-smi

# Install dependencies
!pip install -q fastapi uvicorn python-multipart pyngrok pillow pydantic

# Install llama-cpp-python with CUDA support
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python

print("✅ Dependencies installed!")

## Cell 2: Configure ngrok

Set up the ngrok authentication token using Kaggle secrets.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

# Get token from Kaggle secrets
try:
    user_secrets = UserSecretsClient()
    NGROK_TOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")
    print("✅ Loaded NGROK_AUTHTOKEN from Kaggle secrets")
except Exception as e:
    print(f"⚠️ Could not load from secrets: {e}")
    NGROK_TOKEN = input("Enter your ngrok authtoken: ").strip()

if not NGROK_TOKEN:
    raise ValueError("ngrok authtoken is required!")

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok configured!")

## Cell 3: Download Gemma 3 4B Model

Downloads the quantized Gemma 3 4B model. Kaggle has faster internet than Colab.

In [ ]:
import os

# Model configuration
MODEL_URL = "https://huggingface.co/lmstudio-community/gemma-3-4b-it-GGUF/resolve/main/gemma-3-4b-it-Q4_K_M.gguf"
MODEL_FILENAME = "gemma-3-4b-it-Q4_K_M.gguf"
MODEL_PATH = f"/kaggle/working/{MODEL_FILENAME}"

# Download if not exists
if not os.path.exists(MODEL_PATH):
    print(f"⬇️ Downloading {MODEL_FILENAME}...")
    !wget -q --show-progress {MODEL_URL} -O {MODEL_PATH}
    print("✅ Download complete!")
else:
    print(f"✅ Model already exists at {MODEL_PATH}")

print(f"📦 Model size: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB")

## Cell 4: Load Model & Warmup

Load the model into GPU memory and run warmup inference.

In [ ]:
from llama_cpp import Llama

print("🔄 Loading Gemma 3 4B model...")
print("   Kaggle's GPU has 30GB RAM - plenty of space!")

# Load model with full GPU acceleration
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,          # Context window
    n_gpu_layers=-1,     # Use all layers on GPU
    verbose=False
)

print("✅ Model loaded!")

# Warmup inference
print("🔥 Running warmup inference...")
_ = llm("Say 'ready'", max_tokens=10)
print("✅ Warmup complete! Server is ready.")

## Cell 5: Create FastAPI Server

Create the FastAPI application with health and identification endpoints.

In [ ]:
from fastapi import FastAPI, HTTPException, Header
from pydantic import BaseModel
from typing import List, Optional
from PIL import Image
import io
import base64
import re

app = FastAPI(title="VisionTracker Remote ID Server (Kaggle)", version="1.0.0")

# Optional API key for security
try:
    SERVER_API_KEY = user_secrets.get_secret("SERVER_API_KEY")
except:
    SERVER_API_KEY = None

class IdentifyRequest(BaseModel):
    images: List[str]  # List of base64-encoded JPEG images
    class_names: List[str]  # Corresponding detector class names
    track_ids: List[int]  # Track IDs for response mapping

class IdentifyResponse(BaseModel):
    results: List[dict]

def create_prompt(class_names: List[str]) -> str:
    n = len(class_names)
    if n == 1:
        return (
            f"The object detector classified this as '{class_names[0]}'. "
            "Describe the object in ONE concise sentence (≤ 12 words): "
            "color, shape, key features."
        )
    else:
        numbered = "\n".join(f"{i+1}. <description>" for i in range(n))
        return (
            f"I will show you {n} object crops. The detector classes are: {', '.join(class_names)}.\n"
            "Write ONE short sentence per object (≤ 12 words: color, shape, features).\n"
            f"Reply ONLY in this exact format:\n{numbered}"
        )

def identify_batch(images_b64: List[str], class_names: List[str]) -> List[str]:
    n = len(images_b64)
    prompt = create_prompt(class_names)
    
    content = [{"type": "text", "text": prompt}]
    
    for i, img_b64 in enumerate(images_b64):
        if n > 1:
            content.append({
                "type": "text",
                "text": f"Image {i+1} — detected class: {class_names[i]}"
            })
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
        })
    
    messages = [{"role": "user", "content": content}]
    
    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=100 * n,
        temperature=0.2,
        stop=["</s>"]
    )
    
    raw_text = response["choices"][0]["message"]["content"].strip()
    
    # Parse numbered list
    results = []
    pattern = re.compile(r"^\s*(\d+)[.)]\s*(.+)$")
    
    if n == 1:
        clean = re.sub(r"^1[.)]\s*", "", raw_text).strip()
        results = [clean or class_names[0]]
    else:
        numbered = {}
        for line in raw_text.splitlines():
            m = pattern.match(line)
            if m:
                idx = int(m.group(1)) - 1
                if 0 <= idx < n:
                    numbered[idx] = m.group(2).strip()
        
        for i in range(n):
            if i in numbered:
                results.append(numbered[i])
            else:
                fallback_lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
                results.append(fallback_lines[i] if i < len(fallback_lines) else class_names[i])
    
    return results

@app.get("/health")
async def health_check():
    return {
        "status": "healthy",
        "model": "gemma-3-4b-it",
        "version": "1.0.0",
        "platform": "kaggle"
    }

@app.post("/identify", response_model=IdentifyResponse)
async def identify(
    request: IdentifyRequest,
    authorization: Optional[str] = Header(None)
):
    if SERVER_API_KEY and authorization != f"Bearer {SERVER_API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid or missing API key")
    
    if len(request.images) != len(request.class_names) or len(request.images) != len(request.track_ids):
        raise HTTPException(status_code=400, detail="Mismatched array lengths")
    
    if len(request.images) == 0:
        return IdentifyResponse(results=[])
    
    if len(request.images) > 16:
        raise HTTPException(status_code=400, detail="Batch size too large (max 16)")
    
    try:
        descriptions = identify_batch(request.images, request.class_names)
        results = [
            {"track_id": tid, "description": desc}
            for tid, desc in zip(request.track_ids, descriptions)
        ]
        return IdentifyResponse(results=results)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("✅ FastAPI app created!")
print(f"   API Key required: {SERVER_API_KEY is not None}")

## Cell 6: Start Server with ngrok Tunnel

Starts the ngrok tunnel and FastAPI server. Keep this notebook running!

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

# Create tunnel
public_url = ngrok.connect(8000, "http")
print("━" * 60)
print("🚀 KAGGLE SERVER IS RUNNING!")
print("━" * 60)
print(f"📡 Public URL: {public_url}")
print(f"   Health: {public_url}/health")
print(f"   Identify: {public_url}/identify")
if SERVER_API_KEY:
    print(f"🔑 API Key: {SERVER_API_KEY[:8]}...")
else:
    print("🔓 No API key configured")
print("━" * 60)
print("\n💡 VisionTracker usage:")
print(f"   python main.py --use-remote-gemma --remote-gemma-url {public_url}")
if SERVER_API_KEY:
    print(f"   --remote-gemma-key {SERVER_API_KEY}")
print("\n⚠️  Keep this notebook running!")
print("━" * 60)

uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

---

## 📝 Kaggle-Specific Notes

### Session Limits
- **GPU quota**: 30 hours per week (T4/P100)
- **Session length**: Up to 12 hours continuous
- **Idle timeout**: ~60 minutes (keep browser active)

### Storage
- **Working directory**: Persistent across runs in same notebook version
- **Model is downloaded to**: `/kaggle/working/`
- Clear output files if you need disk space

### Internet Access
- Required for ngrok tunnel
- Enable in notebook settings: Settings → Internet → On

### Performance Tips
- Kaggle's T4 GPU gives ~2-3x faster inference than Colab
- First request after loading is slow (5-10s)
- Subsequent requests: 1-2 seconds

### Troubleshooting
**"No module named llama_cpp"**: Re-run Cell 1

**Out of memory**: Reduce `n_ctx` to 2048 in Cell 4

**Tunnel not connecting**: Verify internet is enabled in notebook settings